# 01 — Data Loading

**Phase 1 — Data Engineering | MVP ladder: pre-MVP**

**Goal:** Parse `MMHS150K_GT.json` into a unified `pandas.DataFrame` with one row per tweet and derive the supervised targets:

- **T1** — binary hate flag (majority of 3 annotators is non-zero)
- **T2** — 6-class hate category (majority vote; on 3-way tie, lowest-index hate class)
- **T3** — annotator agreement score in {0.333, 0.667, 1.0}

**Output:** `data/processed/labels_parsed.csv` with columns
`tweet_id, tweet_text, img_filename, labels, labels_str, T1, T2, T3`.

Structured-feature engineering is deferred to `03_structured_features.ipynb`.
Split membership attachment is deferred to `04_train_val_test_split.ipynb`.

---

### Locked decisions (do not re-litigate)

- **T2 ambiguity** — when `T3 == 0.333` (3-way disagreement), `T2` will be set to NaN and a `t2_valid` boolean column added. T1 and T3 stay valid for these rows. Masked CE used during T2 training. *(Applied after the 22-row anomaly is resolved.)*
- **Religion class (~0.3%)** — reflects real Twitter distribution, not measurement error. No over-sampling or synthetic augmentation. Focal Loss + class-weighted sampling within hate categories is the only mitigation. Low-recall expectation documented as a known limitation.
- **Official splits kept as-is** — train 134,823 / val 5,000 / test 10,000. The val:test 1:2 ratio is by Gomez 2019 design (smaller val, larger test for more reliable test metrics). Not re-balanced.

---

**Content warning:** MMHS150K contains real Twitter posts including slurs and explicit harassment. Do not display raw tweet text in any public-facing artefact. Internal exploration only.


In [1]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Determinism (per CLAUDE.md Section 11)
random.seed(42)
np.random.seed(42)

# Project root anchored relative to this notebook
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "MMHS150K"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

GT_JSON = DATA_DIR / "MMHS150K_GT.json"
IMG_DIR = DATA_DIR / "img_resized"
OCR_DIR = DATA_DIR / "img_txt"
SPLITS_DIR = DATA_DIR / "splits"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("GT_JSON exists:", GT_JSON.exists())
print("IMG_DIR exists:", IMG_DIR.exists())
print("OCR_DIR exists:", OCR_DIR.exists())
print("SPLITS_DIR exists:", SPLITS_DIR.exists())
print("pandas:", pd.__version__, "| numpy:", np.__version__)

PROJECT_ROOT: /teamspace/studios/this_studio/HateFusion
GT_JSON exists: True
IMG_DIR exists: True
OCR_DIR exists: True
SPLITS_DIR exists: True
pandas: 3.0.3 | numpy: 2.4.4


In [2]:
# Cell 3 — Parse GT JSON, build DataFrame, derive T1/T2/T3
from collections import Counter

print(f"Loading {GT_JSON.name} (~270 MB) ...")
with open(GT_JSON, "r", encoding="utf-8") as f:
    gt = json.load(f)
print(f"Loaded {len(gt):,} tweet entries from GT JSON")

# Flatten dict-of-dicts into list of records
records = []
for tweet_id, fields in gt.items():
    records.append({
        "tweet_id":     tweet_id,
        "tweet_text":   fields["tweet_text"],
        "img_filename": f"{tweet_id}.jpg",
        "labels":       fields["labels"],
        "labels_str":   fields["labels_str"],
    })
df = pd.DataFrame(records)

# Free the raw dict — 600+ MB of Python objects we no longer need
del gt, records

print("\nAfter flatten -> df.shape:", df.shape)
print("Nulls per column:")
print(df.isna().sum())

# Derive T1, T2, T3 from the 3-annotator label arrays.
# Tie-break (per CLAUDE.md sec 10): on 3-way tie, prefer lowest-index *hate* class
# (1..5) over NotHate (0). T1 = 1 iff T2 != 0. T3 = max_count / 3.
def derive_labels(labels):
    counter = Counter(labels)
    max_count = max(counter.values())
    top = sorted(c for c, cnt in counter.items() if cnt == max_count)
    if len(top) == 1:
        t2 = top[0]
    else:
        hate_in_tie = [c for c in top if c != 0]
        t2 = hate_in_tie[0] if hate_in_tie else 0
    t1 = 0 if t2 == 0 else 1
    t3 = max_count / 3.0
    return t1, t2, t3

derived = df["labels"].apply(derive_labels)
df[["T1", "T2", "T3"]] = pd.DataFrame(derived.tolist(), index=df.index)
df["T1"] = df["T1"].astype("int8")
df["T2"] = df["T2"].astype("int8")
df["T3"] = df["T3"].astype("float32")

print("\nAfter target derivation -> df.shape:", df.shape)
print("Nulls per column:")
print(df.isna().sum())
print("\ndtypes:")
print(df.dtypes)

# --- Distributions ---
T2_LABELS = {0: "NotHate", 1: "Racist", 2: "Sexist", 3: "Homophobe", 4: "Religion", 5: "OtherHate"}

print("\n--- T1 distribution (binary hate) ---")
t1_counts = df["T1"].value_counts().sort_index()
t1_pct    = df["T1"].value_counts(normalize=True).sort_index().round(4)
print(pd.DataFrame({"count": t1_counts, "fraction": t1_pct}).rename(index={0: "no_hate", 1: "hate"}))

print("\n--- T2 distribution (6-class hate category) ---")
t2_counts = df["T2"].value_counts().sort_index()
t2_pct    = df["T2"].value_counts(normalize=True).sort_index().round(4)
t2_table  = pd.DataFrame({"count": t2_counts, "fraction": t2_pct})
t2_table.index = t2_table.index.map(T2_LABELS)
print(t2_table)

print("\n--- T3 distribution (annotator agreement) ---")
t3_counts = df["T3"].round(3).value_counts().sort_index()
t3_pct    = df["T3"].round(3).value_counts(normalize=True).sort_index().round(4)
print(pd.DataFrame({"count": t3_counts, "fraction": t3_pct}))
print(f"\nT3 mean: {df['T3'].mean():.4f}  |  T3 median: {df['T3'].median():.4f}")

# Sanity vs Gomez 2019 paper (rough expected fractions)
hate_pct = (df["T1"] == 1).mean() * 100
print(f"\nHate (T1=1) fraction: {hate_pct:.2f}% — Gomez 2019 reported approx 32-36%")


Loading MMHS150K_GT.json (~270 MB) ...


Loaded 149,823 tweet entries from GT JSON



After flatten -> df.shape: (149823, 5)
Nulls per column:
tweet_id        0
tweet_text      0
img_filename    0
labels          0
labels_str      0
dtype: int64



After target derivation -> df.shape: (149823, 8)
Nulls per column:
tweet_id        0
tweet_text      0
img_filename    0
labels          0
labels_str      0
T1              0
T2              0
T3              0
dtype: int64

dtypes:
tweet_id            str
tweet_text          str
img_filename        str
labels           object
labels_str       object
T1                 int8
T2                 int8
T3              float32
dtype: object

--- T1 distribution (binary hate) ---
          count  fraction
T1                       
no_hate  112845    0.7532
hate      36978    0.2468

--- T2 distribution (6-class hate category) ---
            count  fraction
T2                         
NotHate    112845    0.7532
Racist      18802    0.1255
Sexist       6944    0.0463
Homophobe    5026    0.0335
Religion      395    0.0026
OtherHate    5811    0.0388

--- T3 distribution (annotator agreement) ---
       count  fraction
T3                    
0.333  11714    0.0782
0.667  76092    0.5079
1.000

In [3]:
# Cell 4 — Persist parsed labels DataFrame to data/processed/labels_parsed.csv
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = PROCESSED_DIR / "labels_parsed.csv"

df.to_csv(OUT_CSV, index=False)
size_mb = OUT_CSV.stat().st_size / 1024 / 1024
print(f"Wrote {len(df):,} rows -> {OUT_CSV}")
print(f"File size: {size_mb:.1f} MB")

# Note: CSV serialises `labels` and `labels_str` as Python repr strings
# (e.g. "[4, 1, 3]"). To recover the lists on read use ast.literal_eval:
#     import ast
#     df_back = pd.read_csv(OUT_CSV, converters={
#         "labels":     ast.literal_eval,
#         "labels_str": ast.literal_eval,
#     })
print("\nFirst 3 rows preview:")
print(df.head(3).to_string())


Wrote 149,823 rows -> /teamspace/studios/this_studio/HateFusion/data/processed/labels_parsed.csv
File size: 27.4 MB

First 3 rows preview:
              tweet_id                                                                        tweet_text             img_filename     labels                         labels_str  T1  T2        T3
0  1114679353714016256                                      @FriskDontMiss Nigga https://t.co/cAsaLWEpue  1114679353714016256.jpg  [4, 1, 3]      [Religion, Racist, Homophobe]   1   1  0.333333
1  1063020048816660480                                    My horses are retarded https://t.co/HYhqc6d5WN  1063020048816660480.jpg  [5, 5, 5]  [OtherHate, OtherHate, OtherHate]   1   5  1.000000
2  1108927368075374593  “NIGGA ON MA MOMMA YOUNGBOY BE SPITTING REAL SHIT NIGGA” https://t.co/UczofqHrLq  1108927368075374593.jpg  [0, 0, 0]        [NotHate, NotHate, NotHate]   0   0  1.000000


In [4]:
# Cell 5 — Investigate anomalous rows where T3 > 1.0
# These are tweets whose `labels` array has more than the documented 3 entries.
# Goal: see what they look like before deciding drop / cap-to-3 / keep.

print("--- Length distribution of `labels` array across ALL rows (sanity) ---")
all_lens = df["labels"].apply(len)
print(all_lens.value_counts().sort_index().rename("count"))

anomalous = df[df["T3"] > 1.0].copy()
anomalous["labels_len"] = anomalous["labels"].apply(len)

print(f"\nTotal anomalous rows (T3 > 1.0): {len(anomalous)}")
print("\n--- Length distribution among anomalies ---")
print(anomalous["labels_len"].value_counts().sort_index().rename("count"))

print("\n--- Per-row detail of every anomalous tweet ---")
for _, row in anomalous.sort_values("labels_len").iterrows():
    text_preview = (row["tweet_text"] or "").replace("\n", " ").replace("\r", " ")[:100]
    print(f"tweet_id={row['tweet_id']:>20}  len={row['labels_len']:>2}  "
          f"T1={row['T1']}  T2={row['T2']}  T3={row['T3']:.3f}")
    print(f"  labels    : {row['labels']}")
    print(f"  labels_str: {row['labels_str']}")
    print(f"  text      : {text_preview}")
    print()


--- Length distribution of `labels` array across ALL rows (sanity) ---
labels
1         4
2        30
3    149749
4        37
5         3
Name: count, dtype: int64

Total anomalous rows (T3 > 1.0): 22

--- Length distribution among anomalies ---
labels_len
4    19
5     3
Name: count, dtype: int64

--- Per-row detail of every anomalous tweet ---
tweet_id= 1109469766450900992  len= 4  T1=0  T2=0  T3=1.333
  labels    : [0, 0, 0, 0]
  labels_str: ['NotHate', 'NotHate', 'NotHate', 'NotHate']
  text      : @gazhursthouse  smelly twat https://t.co/JOjExEkwxM

tweet_id= 1106933213946089473  len= 4  T1=0  T2=0  T3=1.333
  labels    : [0, 0, 0, 0]
  labels_str: ['NotHate', 'NotHate', 'NotHate', 'NotHate']
  text      : “Me I want my pockets fat about a bitch, tired of seein' niggas flaunt, I wanna flaunt too nigga! ht

tweet_id= 1113953819497512963  len= 4  T1=0  T2=0  T3=1.333
  labels    : [0, 0, 0, 0]
  labels_str: ['NotHate', 'NotHate', 'NotHate', 'NotHate']
  text      : “I’ll date the br

In [5]:
# Cell 6 — Apply locked decisions: fix anomalies, mask ambiguous T2, re-save
#
# Per user's locked rules (do not re-litigate):
#   1. Drop the 4 rows with len(labels) == 1 (single annotator, no agreement signal possible)
#   2. Generalise T3 = max_count / len(labels) so T3 always lies in [0, 1]
#   3. Add `n_annotators` column = len(labels)  (lets future sessions see extended annotations)
#   4. Set T2 = NaN where there is no majority (max_count == 1)
#      For len=3 this is equivalent to T3 == 0.333 ("3-way disagreement").
#      For len=2/4/5 it generalises semantically (no class reached >=2 votes).
#   5. Add t2_valid boolean column (True iff T2 is not NaN)
#   6. Re-save labels_parsed.csv with the corrected schema

before_n = len(df)
df["n_annotators"] = df["labels"].apply(len).astype("int8")

# --- Step 1: Drop len=1 rows ---
mask_drop = df["n_annotators"] == 1
print(f"Dropping {mask_drop.sum()} rows with len(labels) == 1 (no agreement signal possible)")
df = df[~mask_drop].copy()
print(f"Rows: {before_n:,} -> {len(df):,}")

# --- Step 2: Recompute T3 with generalised formula ---
def recompute_t3(labels):
    return max(Counter(labels).values()) / len(labels)
df["T3"] = df["labels"].apply(recompute_t3).astype("float32")

# --- Step 3: Verify T3 bounds ---
assert df["T3"].between(0.0, 1.0).all(), \
    f"T3 out of bounds: min={df['T3'].min()}, max={df['T3'].max()}"
print(f"T3 range after fix: [{df['T3'].min():.4f}, {df['T3'].max():.4f}] -- OK")

# --- Step 4 & 5: Compute t2_valid, set T2 = NaN where invalid ---
df["t2_valid"] = df["labels"].apply(lambda x: max(Counter(x).values()) >= 2)
# Convert T2 to nullable Int8 dtype so NaN can be assigned without losing int semantics
df["T2"] = df["T2"].astype("Int8")
df.loc[~df["t2_valid"], "T2"] = pd.NA

# ---------- Verification block ----------
print("\n--- Final shape ---", df.shape)

print("\n--- n_annotators distribution ---")
print(df["n_annotators"].value_counts().sort_index().rename("count"))

print("\n--- T3 distribution (rounded to 3dp, top buckets) ---")
print(df["T3"].round(3).value_counts().sort_index().head(20).rename("count"))
print(f"T3 mean: {df['T3'].mean():.4f}  |  T3 max: {df['T3'].max():.4f}")

T2_LABELS = {0: "NotHate", 1: "Racist", 2: "Sexist", 3: "Homophobe", 4: "Religion", 5: "OtherHate"}
print("\n--- T2 distribution (NaN-aware) ---")
t2_counts = df["T2"].value_counts(dropna=False).sort_index()
# rename int keys; leave NaN row untouched
t2_counts.index = [T2_LABELS.get(k, "<NaN>") if pd.notna(k) else "<NaN>" for k in t2_counts.index]
print(t2_counts.rename("count"))

print(f"\nt2_valid=True : {df['t2_valid'].sum():,}")
print(f"t2_valid=False: {(~df['t2_valid']).sum():,}")

# Null audit
print("\n--- Null audit (per column) ---")
print(df.isna().sum())

t2_nan_count     = int(df["T2"].isna().sum())
t2_invalid_count = int((~df["t2_valid"]).sum())
assert t2_nan_count == t2_invalid_count, \
    f"T2 NaN ({t2_nan_count}) != t2_valid=False ({t2_invalid_count})"
print(f"\nT2 NaN count ({t2_nan_count:,}) matches t2_valid=False count ({t2_invalid_count:,}) -- OK")

# Confirm only T2 has nulls
nulls_outside_t2 = df.drop(columns=["T2"]).isna().sum()
assert nulls_outside_t2.sum() == 0, f"Unexpected nulls outside T2:\n{nulls_outside_t2}"
print("Only T2 has nulls; T1, T3, n_annotators, t2_valid all clean -- OK")

# --- Save ---
df.to_csv(OUT_CSV, index=False)
size_mb = OUT_CSV.stat().st_size / 1024 / 1024
print(f"\nSaved {len(df):,} rows -> {OUT_CSV}  ({size_mb:.1f} MB)")


Dropping 4 rows with len(labels) == 1 (no agreement signal possible)
Rows: 149,823 -> 149,819


T3 range after fix: [0.3333, 1.0000] -- OK



--- Final shape --- (149819, 10)

--- n_annotators distribution ---
n_annotators
2        30
3    149749
4        37
5         3
Name: count, dtype: int64

--- T3 distribution (rounded to 3dp, top buckets) ---
T3
0.333    11699
0.500       16
0.667    76068
0.750       13
0.800        2
1.000    62021
Name: count, dtype: int64
T3 mean: 0.7786  |  T3 max: 1.0000

--- T2 distribution (NaN-aware) ---
NotHate      112842
Racist        11927
Sexist         3495
Homophobe      3871
Religion        163
OtherHate      5811
<NaN>         11710
Name: count, dtype: Int64

t2_valid=True : 138,109
t2_valid=False: 11,710

--- Null audit (per column) ---
tweet_id            0
tweet_text          0
img_filename        0
labels              0
labels_str          0
T1                  0
T2              11710
T3                  0
n_annotators        0
t2_valid            0
dtype: int64

T2 NaN count (11,710) matches t2_valid=False count (11,710) -- OK
Only T2 has nulls; T1, T3, n_annotators, t2_valid a


Saved 149,819 rows -> /teamspace/studios/this_studio/HateFusion/data/processed/labels_parsed.csv  (28.4 MB)
